In [1]:
! pip install confluent_kafka
! pip install pyspark==3.5.2
! pip install elasticsearch

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 5.2 MB/s  0:00:01 eta 0:00:01
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 10.1 MB/s  0:00:30:00:0100:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812426 sha256=2437ae43f52ae81d2c356f47cf0366d4264c1635953f435c819aa6030cc667f4
  Stored in directory: /Users/josepha/Library/Caches/pip/wheels/11/67/ea/33c283e520b775aa7a7a0d404447e287be841a711d074d4d91
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pysp

In [3]:
# ============================================================
# TODO在这里: Sentiment Analysis 可以用多个模型来对比
# ============================================================

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def analyze_sentiment(text):
    if not text:
        return "neutral"
    # TODO: Replace with actual sentiment analysis
    return "neutral"

sentiment_udf = udf(analyze_sentiment, StringType())


In [37]:
# # ============================================================
# # 清空 Checkpoint, 从这个目录是用来记录读到哪一条了
# # ============================================================
# import shutil

# # 删除整个 checkpoint 目录
# checkpoint_dir = "/kaggle/working/checkpoints"
# shutil.rmtree(checkpoint_dir, ignore_errors=True)

# # 也删掉旧的 checkpoint.txt（如果存在）
# import os
# if os.path.exists("/kaggle/working/checkpoint.txt"):
#     os.remove("/kaggle/working/checkpoint.txt")

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from elasticsearch import Elasticsearch, helpers
import json
import os

# ============================================================
# Configuration
# ============================================================

config = {
    "kafka": {
        "bootstrap.servers": "pkc-ox31np.ap-southeast-7.aws.confluent.cloud:9092",
        "security.protocol": "SASL_SSL",
        "sasl.mechanisms": "PLAIN",
        "sasl.username": "S4AU73FQKMOAMSPL",
        "sasl.password": "cfltEuKzwF9VDz4Y2XwilhMpyc+9rxUP4xYfpOFYbbC1VefzzNc8rOYHfIavyVLw",
    },
    "elasticsearch": {
        "host": "https://20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com:443",
        "api_key": "eEFjNGJaMEJfQXBJNldXZFc3YU06RWhYdmpGZ1lsNzZlYWJJa1QwMXdQdw==",
        "index": "enron_emails"
    }
}

# Checkpoint directory
checkpoint_dir = "./checkpoints/enron_kafka_es"
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# ============================================================
# Schema for Enron emails (matches your Kafka data)
# ============================================================

email_schema = StructType([
    StructField("file_path", StringType(), True),
    StructField("message_id", StringType(), True),
    StructField("sent_at", StringType(), True),
    StructField("sender", StringType(), True),
    StructField("to", StringType(), True),
    StructField("cc", StringType(), True),
    StructField("bcc", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("x_folder", StringType(), True),
    StructField("x_origin", StringType(), True),
    StructField("x_filename", StringType(), True),
    StructField("body", StringType(), True),
    StructField("body_length", IntegerType(), True)
])

In [6]:
# ============================================================
# 清空已有的 Elasticsearch 索引
# ============================================================
es_client = Elasticsearch(
    config['elasticsearch']['host'],
    api_key=config['elasticsearch']['api_key'],
    verify_certs=False
)

index_name = "enron_emails"

if es_client.indices.exists(index=index_name):
    # 删除索引
    es_client.indices.delete(index=index_name)
    print(f"Deleted index: {index_name}")
else:
    print(f"Index {index_name} does not exist")


/Users/josepha/Library/Python/3.9/lib/python/site-packages/elasticsearch/_sync/client/__init__.py:312: SecurityWarning: Connecting to 'https://20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com:443' using TLS with verify_certs=False is insecure
  _transport = transport_class(
/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthed

Deleted index: enron_emails


In [7]:

# ============================================================
# Elasticsearch client (for manual bulk writes)
# ============================================================
es_client = Elasticsearch(
    config['elasticsearch']['host'],
    api_key=config['elasticsearch']['api_key'],
    verify_certs=False
)

index_name = "enron_emails"

# create index mapping
mapping = {
    "mappings": {
        "properties": {
            "bcc": {"type": "keyword"},
            "body": {"type": "text"},
            "body_length": {"type": "integer"},
            "cc": {"type": "keyword"},
            "file_path": {"type": "keyword"},
            "message_id": {"type": "keyword"},
            "sender": {"type": "keyword"},
            "sent_at": {
                "type": "date",
                "format": "yyyy-MM-dd HH:mm:ss||yyyy-MM-dd'T'HH:mm:ss.SSSZ||strict_date_optional_time||epoch_millis"
            },
            "sentiment": {"type": "keyword"},
            "subject": {
                "type": "text",
                "fields": {
                    "keyword": {
                        "type": "keyword",
                        "ignore_above": 256
                    }
                }
            },
            "to": {"type": "keyword"},
            "x_filename": {"type": "keyword"},
            "x_folder": {"type": "keyword"},
            "x_origin": {"type": "keyword"}
        }
    }
}

es_client.indices.create(index=index_name, body=mapping)

# # test
# test_doc = {
#     "sent_at": "2001-05-14 23:39:00",
#     "message_id": "test_123",
#     "sender": "test@enron.com",
#     "to": "test@enron.com",
#     "subject": "Test",
#     "body": "Test message",
#     "sentiment": "neutral"
# }

# result = es_client.index(index=index_name, document=test_doc, refresh=True)
# print(f"Test passed! Document ID: {result['_id']}")

/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'enron_emails'})

In [8]:
def write_to_es(df, epoch_id):
    if df.count() == 0:
        return
    
    # 转换 sent_at 日期格式
    if 'sent_at' in df.columns:
        from pyspark.sql.functions import regexp_replace, to_timestamp, date_format
        df = df.withColumn('sent_at', 
            regexp_replace(col('sent_at'), ' UTC$', ''))
        df = df.withColumn('sent_at', 
            to_timestamp(col('sent_at'), 'yyyy-MM-dd HH:mm:ss'))
        df = df.withColumn('sent_at',
            date_format(col('sent_at'), "yyyy-MM-dd'T'HH:mm:ss.SSSZ"))
    
    records = df.toJSON().collect()
    actions = []
    for record in records:
        actions.append({
            "_index": config['elasticsearch']['index'],
            "_source": json.loads(record)
        })
    
    if actions:
        # print(f"Sample record: {actions[0]['_source']}")
        
        success, failed = helpers.bulk(es_client, actions, stats_only=False, raise_on_error=False)
        print(f"Batch {epoch_id}: {success} succeeded, {len(failed)} failed")
        
        if failed:
            print(f"First failed record: {actions[0]['_source']}")
            print(f"Error details: {failed[0]}")

In [9]:
# ============================================================
# Create Spark Session
# ============================================================

spark = SparkSession.builder \
    .appName("EnronKafkaToES") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.2") \
    .getOrCreate()

# 设置日志级别为 ERROR 或 FATAL 来屏蔽 WARN
spark.sparkContext.setLogLevel("ERROR")

# ============================================================
# Read from Kafka and write to Elasticsearch
# ============================================================

def read_from_kafka_and_write_to_es(spark):
    topic = "raw_emails_topic"
    
    stream_df = (spark.readStream
                 .format("kafka")
                 .option("kafka.bootstrap.servers", config['kafka']['bootstrap.servers'])
                 .option("subscribe", topic)
                 .option("startingOffsets", "earliest")
                 .option("kafka.security.protocol", config['kafka']['security.protocol'])
                 .option("kafka.sasl.mechanism", config['kafka']['sasl.mechanisms'])
                 .option("kafka.sasl.jaas.config",
                        f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{config["kafka"]["sasl.username"]}" '
                        f'password="{config["kafka"]["sasl.password"]}";')
                 .option("failOnDataLoss", "false")
                 .option("maxOffsetsPerTrigger", 100)
                 .load()
                )
    
    # Parse JSON
    parsed_df = stream_df.select(
        from_json(col('value').cast("string"), email_schema).alias("data")
    ).select("data.*")
    
    # Add sentiment (placeholder)
    enriched_df = parsed_df.withColumn("sentiment", sentiment_udf(col("body")))
    
    # Write to Elasticsearch via foreachBatch
    query = (enriched_df.writeStream
             .foreachBatch(write_to_es)
             .outputMode("append")
             .trigger(processingTime="10 seconds")
             .option("checkpointLocation", checkpoint_dir)
             .start()
             .awaitTermination()
            )

26/04/11 17:50:39 WARN Utils: Your hostname, JosephadeMacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.2.66 instead (on interface en0)
26/04/11 17:50:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/josepha/.ivy2/cache
The jars for the packages stored in: /Users/josepha/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3a27394c-0a50-4cce-9988-7d5d3cbeaf94;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/josepha/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.2 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.2/spark-sql-kafka-0-10_2.12-3.5.2.jar ...
	[SUCCESSFUL ] org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.2!spark-sql-kafka-0-10_2.12.jar (1092ms)
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_2.12/3.5.2/spark-token-provider-kafka

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50358)
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/josepha/Library/Python/3.9/lib/python/

In [10]:
# 一直运行就会一直流处理    
if __name__ == "__main__":
    read_from_kafka_and_write_to_es(spark)

/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 0: 96 succeeded, 0 failed


/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 1: 96 succeeded, 0 failed


/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

Batch 448: 96 succeeded, 0 failed


/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Batch 449: 96 succeeded, 0 failed


26/04/11 19:41:38 ERROR PythonUDFRunner: Python worker exited unexpectedly (crashed)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/Users/josepha/Library/Python/3.9/lib/python/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1225, in main
    eval_type = read_int(infile)
  File "/Users/josepha/Library/Python/3.9/lib/python/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 596, in read_int
    raise EOFError
EOFError

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)


StreamingQueryException: [STREAM_FAILED] Query [id = 7c904b74-8419-4212-9651-2594ac2a7611, runId = 0edb3fc7-8721-4334-bf0c-bba69e6bb43c] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/Users/josepha/Library/Python/3.9/lib/python/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/Users/josepha/Library/Python/3.9/lib/python/site-packages/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/Users/josepha/Library/Python/3.9/lib/python/site-packages/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/var/folders/lx/hsv_3xp96zqcw0l5z7tx5pkh0000gn/T/ipykernel_34351/3984145864.py", line 2, in write_to_es
    if df.count() == 0:
  File "/Users/josepha/Library/Python/3.9/lib/python/site-packages/pyspark/sql/dataframe.py", line 1240, in count
    return int(self._jd

In [12]:
# 查询最新的5条记录
response = es_client.search(
    index="enron_emails",
    body={
        "query": {"match_all": {}},
        "sort": [{"sent_at": {"order": "desc"}}],
        "size": 5
    }
)

# 打印结果
print(f"Total records: {response['hits']['total']['value']}")
print("\n=== Latest 5 Records ===\n")
for i, hit in enumerate(response['hits']['hits'], 1):
    source = hit['_source']
    print(f"{i}. ID: {hit['_id']}")
    print(f"   Sent at: {source.get('sent_at')}")
    print(f"   From: {source.get('sender')}")
    print(f"   To: {source.get('to')}")
    print(f"   Subject: {source.get('subject', '')[:50]}")
    print(f"   Sentiment: {source.get('sentiment')}")
    print("-" * 50)

Total records: 10000

=== Latest 5 Records ===

1. ID: GjUMfJ0BpeIanCvLjcUE
   Sent at: 2004-02-04T01:45:38.000+0800
   From: 1800flowers.224433405@s2u2.com
   To: ebass@enron.com
   Subject: SAVE $20* while sending gifts to everyone on your 
   Sentiment: neutral
--------------------------------------------------
2. ID: fTUJfJ0BpeIanCvLy76Y
   Sent at: 2004-02-04T01:45:35.000+0800
   From: 1800flowers.238953685@s2u2.com
   To: ebass@enron.com
   Subject: Treat yourself to savings in our Post-Holiday Sale
   Sentiment: neutral
--------------------------------------------------
3. ID: jDUEfJ0BpeIanCvLA7Bg
   Sent at: 2002-03-21T18:48:12.000+0800
   From: sbailey@crusescott.com
   To: susan.bailey@enron.com
   Subject: FW: Strawnie
   Sentiment: neutral
--------------------------------------------------
4. ID: 3jUDfJ0BpeIanCvLta8N
   Sent at: 2002-03-21T18:11:22.000+0800
   From: susan.bailey@enron.com
   To: louis.dicarlo@enron.com
   Subject: RE: EOTT Energy Partners LP
   Sentiment: n

/Users/josepha/Library/Python/3.9/lib/python/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '20910a71bbfd4502b154930abc1ea8d8.asia-southeast1.gcp.elastic-cloud.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [2]:
# === 检查 ES 是否有新写入（情感/关系边） ===
import time
from elasticsearch import Elasticsearch

ES_HOST = "http://localhost:9200"
INDEX_SENT = "enron_emails_sentiment"
INDEX_EDGES = "enron_edges"

es = Elasticsearch(ES_HOST)

def safe_count(idx):
    try:
        return es.count(index=idx)["count"]
    except Exception as e:
        print(f"[WARN] count {idx} failed: {e}")
        return -1

print("=== Snapshot A ===")
a_sent = safe_count(INDEX_SENT)
a_edge = safe_count(INDEX_EDGES)
print(f"{INDEX_SENT}: {a_sent}")
print(f"{INDEX_EDGES}: {a_edge}")

time.sleep(5)

print("\n=== Snapshot B (after 5s) ===")
b_sent = safe_count(INDEX_SENT)
b_edge = safe_count(INDEX_EDGES)
print(f"{INDEX_SENT}: {b_sent} (delta: {b_sent - a_sent:+d})")
print(f"{INDEX_EDGES}: {b_edge} (delta: {b_edge - a_edge:+d})")

# ==============================
# Sentiment 文档（改成结构化打印）
# ==============================
print("\n=== Latest Sentiment Records (Top 5) ===\n")
try:
    r1 = es.search(
        index=INDEX_SENT,
        size=5,
        sort=[{"ingested_at": {"order": "desc", "unmapped_type": "date"}}],
        _source=["message_id", "sender", "subject", "sentiment", "sentiment_score", "ingested_at"],
    )

    total = r1["hits"]["total"]["value"]
    print(f"Total records: {total}\n")

    for i, h in enumerate(r1["hits"]["hits"], 1):
        s = h["_source"]
        print(f"{i}. ID: {h['_id']}")
        print(f"   Ingested at: {s.get('ingested_at')}")
        print(f"   Sender: {s.get('sender')}")
        print(f"   Sentiment: {s.get('sentiment')} ({s.get('sentiment_score')})")
        print(f"   Subject: {(s.get('subject') or '')[:60]}")
        print("-" * 50)

except Exception as e:
    print(f"[WARN] query {INDEX_SENT} failed: {e}")

# ==============================
# Edge 文档（改成结构化打印）
# ==============================
print("\n=== Latest Edge Records (Top 5) ===\n")
try:
    r2 = es.search(
        index=INDEX_EDGES,
        size=5,
        sort=[{"last_contact_at": {"order": "desc", "unmapped_type": "date"}}],
        _source=["src", "dst", "weight", "avg_sentiment", "last_contact_at", "sample_subject"],
    )

    total = r2["hits"]["total"]["value"]
    print(f"Total records: {total}\n")

    for i, h in enumerate(r2["hits"]["hits"], 1):
        s = h["_source"]
        print(f"{i}. ID: {h['_id']}")
        print(f"   Last contact: {s.get('last_contact_at')}")
        print(f"   From: {s.get('src')}")
        print(f"   To: {s.get('dst')}")
        print(f"   Weight: {s.get('weight')}")
        print(f"   Avg Sentiment: {s.get('avg_sentiment')}")
        print(f"   Sample Subject: {(s.get('sample_subject') or '')[:60]}")
        print("-" * 50)

except Exception as e:
    print(f"[WARN] query {INDEX_EDGES} failed: {e}")

=== Snapshot A ===
enron_emails_sentiment: 65800
enron_edges: 23138

=== Snapshot B (after 5s) ===
enron_emails_sentiment: 65800 (delta: +0)
enron_edges: 23138 (delta: +0)

=== Latest Sentiment Records (Top 5) ===

Total records: 10000

1. ID: <31778968.1075843334524.JavaMail.evans@thyme>
   Ingested at: 2026-04-13T13:18:05.246012+00:00
   Sender: jeff.dasovich@enron.com
   Sentiment: NEGATIVE (0.9988732933998108)
   Subject: New Bill Introduced in CA Legislature
--------------------------------------------------
2. ID: <23226007.1075843334551.JavaMail.evans@thyme>
   Ingested at: 2026-04-13T13:18:05.246118+00:00
   Sender: jeff.dasovich@enron.com
   Sentiment: NEGATIVE (0.9938690066337585)
   Subject: IEPs Latest IEP Blueprint for Non Gas-Fired QF SRAC Energy P
--------------------------------------------------
3. ID: <15206072.1075842955609.JavaMail.evans@thyme>
   Ingested at: 2026-04-13T13:18:05.246130+00:00
   Sender: jeff.dasovich@enron.com
   Sentiment: POSITIVE (0.9942143559455